In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

using Revise

In [ ]:
isCuda = try
    success(`nvidia-smi`)
catch
    false
end

In [ ]:
if isCuda
    println("CUDA is available. Loading CUDA.jl...")
    using CUDA
end

In [ ]:
using bslLD, Plots, Statistics

    
isCuda &&bslLD.use_cuda!()

In [ ]:
mutable struct Diag
    rhoe::Vector
    rhoi::Vector
    fi ::Vector
    fe ::Vector
end
Diag() = Diag([], [], [], [])

function diags!(diags,fi , grid)
    push!(diags.fi, copy(fi.data))
    push!(diags.rhoi, copy(bslLD.compute_density(fi, grid).data[:]))
end



function step!(fi, grid, simTime)
    rhoi = bslLD.compute_density(fi, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi), grid, bslLD.PoissonSolver(-1.0))
    sol.E[1].data .= 1.0
    sol.E[2].data .= 0.0
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fi, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0
    bslLD.advectX!(fi, grid, simTime)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi), grid, bslLD.PoissonSolver(-1.0))
    sol.E[1].data .= 1.0
    sol.E[2].data .= 0.0
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fi, grid, simTime, sol.E)

    simTime.fraction_dt = 1.0
    return rhoi
end


In [ ]:
Lx   = 20.0
Nx   = 8
Nv   = 64
vmax = 8.0

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)
simTime = bslLD.SimulationTime(0.005, 10.0; gyro_frequency=1.0)

# Ion distribution: Maxwellian in (vx, vy) with a random density perturbation in x
initFuncx(x) = 1.0 + epsilon * exp(-(x - Lx/2)^2 / (Lx/10)^2)
initFuncv(v) = exp(-(v)^2 / 2) / sqrt(2pi)

f_i = bslLD.Distribution(grid, 0.0;
    initFuncv = initFuncv,
    q=1
)

f_e = bslLD.Distribution(grid, 0.0;
    initFuncv = initFuncv,
    q=-1
)

In [ ]:
diags_i = Diag()
diags_e = Diag()
while bslLD.continue_advection(simTime,true)
    step!(f_i, grid, simTime)
    step!(f_e, grid, simTime)
    diags!(diags_i, f_i, grid)
    diags!(diags_e, f_e, grid)
    bslLD.advance!(simTime)
end    

In [ ]:
fdiag_i = x-> bslLD.DistributionGrid1d2v{Float64,bslLD.Cart,typeof(x)}(x,1.0,1.0)
fdiag_e = x-> bslLD.DistributionGrid1d2v{Float64,bslLD.Cart,typeof(x)}(x,1.0,-1.0)

In [ ]:
jAnaq(t0, q) = [sin(q * t0), -1 + cos(q * t0)]
ΠAnaq(t0, q) = [(3 - cos(2 * q * t0))/2  (-1 + cos(q * t0)) * sin(q * t0); 
               (-1 + cos(q * t0)) * sin(q * t0)  2 + (-2 + cos(q * t0)) * cos(q * t0)]

jAnaSelect(q) = t-> jAnaq(q,t)
ΠAnaSelect(q) = t-> ΠAnaq(q,t)

In [ ]:
j_ilab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_i.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(j_ilab...)))

J_i = hcat([[jAnaSelect(1)(x)[1], jAnaSelect(1)(x)[2]] for x in simTime]...)';
plot!(collect(simTime), J_i, label=["j₁" "j₂"])


In [ ]:
j_elab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_e.fi) ]
plot(collect(simTime)[2:end], transpose(hcat(j_elab...)))

J_e = hcat([[jAnaSelect(-1)(x)[1], jAnaSelect(-1)(x)[2]] for x in simTime]...)';
plot!(collect(simTime), J_e, label=["j₁" "j₂"])

In [ ]:
pi_lab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_momentum_tensor(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_i.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(pi_lab...)))

ΠAna_i = ΠAnaSelect(1)

M_i = hcat([[ΠAna_i(x)[1,1], ΠAna_i(x)[1,2], ΠAna_i(x)[2,2]] for x in simTime]...)';
plot!(collect(simTime), M_i, label=["P₁₁" "P₁₂" "P₂₂"], linestyle=:dash)


In [ ]:
pi_lab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_momentum_tensor(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_e.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(pi_lab...)))

ΠAna_e = ΠAnaSelect(-1)

M_e = hcat([[ΠAna_e(x)[1,1], ΠAna_e(x)[1,2], ΠAna_e(x)[2,2]] for x in simTime]...)';
plot!(collect(simTime), M_e, label=["P₁₁" "P₁₂" "P₂₂"], linestyle=:dash)

In [ ]:
function norm(x::Vector)
    sum(x .* x)
end 

In [ ]:
## Plot error between numerical and analytical solutions for j and Π

error_j_i = [norm(map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) - jAnaSelect(1)(simTime.dt*i)) for i in 1:length(diags_i.fi) ]

plot(collect(simTime)[2:end], error_j_i, label="Error in j_i", yaxis=:log)

error_j_e = [norm(map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) - jAnaSelect(-1)(simTime.dt*i)) for i in 1:length(diags_e.fi) ]

plot!(collect(simTime)[2:end], error_j_e, label="Error in j_e", yaxis=:log)

